In [ ]:
# ================================================================
# NOTEBOOK 3 — SQL DEEP ANALYSIS
# Project: E-Commerce Intelligence Platform — Olist Brazil
# ================================================================

import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------------------
# CELL 1 — DATABASE SETUP
# ----------------------------------------------------------------
path = "../data_raw/raw/"

orders      = pd.read_csv(path + "olist_orders_dataset.csv")
customers   = pd.read_csv(path + "olist_customers_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
products    = pd.read_csv(path + "olist_products_dataset.csv")
sellers     = pd.read_csv(path + "olist_sellers_dataset.csv")
payments    = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(path + "olist_order_reviews_dataset.csv")
category    = pd.read_csv(path + "product_category_name_translation.csv")
rfm         = pd.read_csv("../outputs/rfm_customers.csv")

# Create SQLite database
conn = sqlite3.connect('../data/ecommerce.db')

orders.to_sql('orders',           conn, if_exists='replace', index=False)
customers.to_sql('customers',     conn, if_exists='replace', index=False)
order_items.to_sql('order_items', conn, if_exists='replace', index=False)
products.to_sql('products',       conn, if_exists='replace', index=False)
sellers.to_sql('sellers',         conn, if_exists='replace', index=False)
payments.to_sql('payments',       conn, if_exists='replace', index=False)
reviews.to_sql('reviews',         conn, if_exists='replace', index=False)
category.to_sql('category',       conn, if_exists='replace', index=False)
rfm.to_sql('rfm_segments',        conn, if_exists='replace', index=False)

print("Database created successfully!")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"Tables: {tables['name'].tolist()}")

# Helper function
def run_query(title, query, save_result=True):
    print(f"\n{'='*65}")
    print(f"{title}")
    print('='*65)
    result = pd.read_sql(query, conn)
    print(result.to_string(index=False))
    return result


# ----------------------------------------------------------------
# CELL 2 — QUERY 1: COMPLETE ORDER FUNNEL
# WHAT: Every stage conversion rate
# WHY:  Find exactly where business loses customers and revenue
# HOW:  Fix highest drop-off stage first
# ----------------------------------------------------------------
q1 = run_query("Q1: COMPLETE ORDER FUNNEL ANALYSIS", """
    SELECT
        order_status,
        COUNT(*)                                              AS order_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2)   AS pct_of_total,
        ROUND(SUM(p.payment_value), 2)                       AS revenue
    FROM orders o
    LEFT JOIN payments p ON o.order_id = p.order_id
    GROUP BY order_status
    ORDER BY order_count DESC
""")


# ----------------------------------------------------------------
# CELL 3 — QUERY 2: SELLER QUALITY SCORECARD
# WHAT: Every seller ranked by performance
# WHY:  Bad sellers = bad reviews = customer churn
# HOW:  Warn bottom sellers, reward top sellers
# ----------------------------------------------------------------
q2 = run_query("Q2: SELLER QUALITY SCORECARD — TOP 15", """
    SELECT
        s.seller_id,
        s.seller_state,
        COUNT(DISTINCT oi.order_id)                          AS total_orders,
        ROUND(SUM(oi.price), 2)                              AS total_revenue,
        ROUND(AVG(r.review_score), 2)                        AS avg_review,
        ROUND(AVG(julianday(o.order_delivered_customer_date)
              - julianday(o.order_purchase_timestamp)), 1)   AS avg_delivery_days,
        ROUND(SUM(CASE WHEN o.order_delivered_customer_date
                            > o.order_estimated_delivery_date
                       THEN 1.0 ELSE 0 END)
              * 100.0 / COUNT(*), 2)                         AS late_pct,
        CASE
            WHEN AVG(r.review_score) >= 4.5
             AND SUM(CASE WHEN o.order_delivered_customer_date
                               > o.order_estimated_delivery_date
                          THEN 1.0 ELSE 0 END)
                 * 100.0 / COUNT(*) < 5                      THEN 'TOP PERFORMER'
            WHEN AVG(r.review_score) < 3.0
              OR SUM(CASE WHEN o.order_delivered_customer_date
                               > o.order_estimated_delivery_date
                          THEN 1.0 ELSE 0 END)
                 * 100.0 / COUNT(*) > 20                     THEN 'AT RISK'
            ELSE 'AVERAGE'
        END                                                  AS seller_grade
    FROM sellers s
    JOIN order_items oi  ON s.seller_id  = oi.seller_id
    JOIN orders o        ON oi.order_id  = o.order_id
    JOIN reviews r       ON o.order_id   = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY s.seller_id, s.seller_state
    HAVING COUNT(DISTINCT oi.order_id) >= 50
    ORDER BY avg_review DESC, total_revenue DESC
    LIMIT 15
""")


# ----------------------------------------------------------------
# CELL 4 — QUERY 3: CUSTOMER RETENTION RATE
# WHAT: Month-wise repeat purchase percentage
# WHY:  Retention = business health indicator
# HOW:  Improve onboarding for months with lowest retention
# ----------------------------------------------------------------
q3 = run_query("Q3: MONTHLY CUSTOMER RETENTION RATE", """
    WITH first_orders AS (
        SELECT
            customer_id,
            MIN(strftime('%Y-%m', order_purchase_timestamp)) AS first_month
        FROM orders
        WHERE order_status = 'delivered'
        GROUP BY customer_id
    ),
    monthly_orders AS (
        SELECT DISTINCT
            o.customer_id,
            strftime('%Y-%m', o.order_purchase_timestamp) AS order_month
        FROM orders o
        WHERE o.order_status = 'delivered'
    )
    SELECT
        f.first_month                                        AS cohort_month,
        COUNT(DISTINCT f.customer_id)                        AS new_customers,
        COUNT(DISTINCT m.customer_id)                        AS returning_customers,
        ROUND(COUNT(DISTINCT m.customer_id) * 100.0
              / COUNT(DISTINCT f.customer_id), 2)            AS retention_rate_pct
    FROM first_orders f
    LEFT JOIN monthly_orders m
           ON f.customer_id = m.customer_id
          AND m.order_month > f.first_month
    GROUP BY f.first_month
    ORDER BY f.first_month
""")


# ----------------------------------------------------------------
# CELL 5 — QUERY 4: REVENUE PER CUSTOMER BY STATE
# WHAT: Not just total revenue — quality of customers per state
# WHY:  Total revenue misleads — SP is big but is it efficient?
# HOW:  Focus expansion on high revenue-per-customer states
# ----------------------------------------------------------------
q4 = run_query("Q4: REVENUE QUALITY BY STATE", """
    SELECT
        c.customer_state                                     AS state,
        COUNT(DISTINCT o.order_id)                           AS total_orders,
        COUNT(DISTINCT o.customer_id)                        AS unique_customers,
        ROUND(SUM(p.payment_value), 2)                       AS total_revenue,
        ROUND(AVG(p.payment_value), 2)                       AS avg_order_value,
        ROUND(SUM(p.payment_value)
              / COUNT(DISTINCT o.customer_id), 2)            AS revenue_per_customer,
        ROUND(AVG(r.review_score), 2)                        AS avg_review
    FROM orders o
    JOIN customers c  ON o.customer_id  = c.customer_id
    JOIN payments p   ON o.order_id     = p.order_id
    JOIN reviews r    ON o.order_id     = r.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
    HAVING COUNT(DISTINCT o.order_id) >= 100
    ORDER BY revenue_per_customer DESC
""")


# ----------------------------------------------------------------
# CELL 6 — QUERY 5: CATEGORY SEASONALITY MATRIX
# WHAT: Which category peaks in which month
# WHY:  Inventory planning + marketing calendar
# HOW:  Stock up 30 days before peak, run campaigns at peak
# ----------------------------------------------------------------
q5 = run_query("Q5: CATEGORY SEASONALITY — TOP 8 CATEGORIES", """
    WITH cat_monthly AS (
        SELECT
            COALESCE(c.product_category_name_english,
                     'Unknown')                              AS category,
            CAST(strftime('%m', o.order_purchase_timestamp)
                 AS INTEGER)                                 AS month_num,
            ROUND(SUM(oi.price), 2)                          AS monthly_revenue
        FROM order_items oi
        JOIN orders o    ON oi.order_id  = o.order_id
        JOIN products p  ON oi.product_id = p.product_id
        LEFT JOIN category c
               ON p.product_category_name = c.product_category_name
        WHERE o.order_status = 'delivered'
        GROUP BY category, month_num
    ),
    cat_total AS (
        SELECT category, SUM(monthly_revenue) AS total_rev
        FROM cat_monthly
        GROUP BY category
        ORDER BY total_rev DESC
        LIMIT 8
    )
    SELECT
        cm.category,
        cm.month_num,
        cm.monthly_revenue,
        ROUND(cm.monthly_revenue * 100.0
              / ct.total_rev, 2)                             AS pct_of_annual
    FROM cat_monthly cm
    JOIN cat_total ct ON cm.category = ct.category
    ORDER BY ct.total_rev DESC, cm.month_num
""")


# ----------------------------------------------------------------
# CELL 7 — QUERY 6: DELIVERY ROOT CAUSE ANALYSIS
# WHAT: Late deliveries by seller + state + product weight
# WHY:  Find exact combination causing most late deliveries
# HOW:  Fix top 3 seller-state combinations = biggest impact
# ----------------------------------------------------------------
q6 = run_query("Q6: DELIVERY ROOT CAUSE — SELLER + STATE + WEIGHT", """
    SELECT
        s.seller_state,
        c.customer_state,
        COUNT(*)                                             AS total_orders,
        ROUND(AVG(oi.product_weight_g), 0)                  AS avg_weight_g,
        ROUND(AVG(julianday(o.order_delivered_customer_date)
              - julianday(o.order_purchase_timestamp)), 1)   AS avg_days,
        ROUND(SUM(CASE WHEN o.order_delivered_customer_date
                            > o.order_estimated_delivery_date
                       THEN 1.0 ELSE 0 END)
              * 100.0 / COUNT(*), 2)                         AS late_pct,
        ROUND(AVG(r.review_score), 2)                        AS avg_review
    FROM orders o
    JOIN order_items oi ON o.order_id   = oi.order_id
    JOIN products p     ON oi.product_id = p.product_id
    JOIN sellers s      ON oi.seller_id  = s.seller_id
    JOIN customers c    ON o.customer_id = c.customer_id
    JOIN reviews r      ON o.order_id    = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY s.seller_state, c.customer_state
    HAVING COUNT(*) >= 100
    ORDER BY late_pct DESC
    LIMIT 15
""")


# ----------------------------------------------------------------
# CELL 8 — QUERY 7: REVIEW PATTERN ANALYSIS
# WHAT: 1-star reviews — which category, seller, delivery combo
# WHY:  1-star = public reputation damage
# HOW:  Fix root cause categories and sellers immediately
# ----------------------------------------------------------------
q7 = run_query("Q7: 1-STAR REVIEW ROOT CAUSE ANALYSIS", """
    SELECT
        COALESCE(cat.product_category_name_english,
                 'Unknown')                                  AS category,
        COUNT(*)                                             AS total_reviews,
        SUM(CASE WHEN r.review_score = 1 THEN 1 ELSE 0 END) AS one_star_count,
        ROUND(SUM(CASE WHEN r.review_score = 1
                       THEN 1.0 ELSE 0 END)
              * 100.0 / COUNT(*), 2)                         AS one_star_pct,
        ROUND(AVG(r.review_score), 2)                        AS avg_review,
        ROUND(AVG(julianday(o.order_delivered_customer_date)
              - julianday(o.order_purchase_timestamp)), 1)   AS avg_delivery_days,
        ROUND(SUM(CASE WHEN o.order_delivered_customer_date
                            > o.order_estimated_delivery_date
                       THEN 1.0 ELSE 0 END)
              * 100.0 / COUNT(*), 2)                         AS late_pct
    FROM reviews r
    JOIN orders o      ON r.order_id    = o.order_id
    JOIN order_items oi ON o.order_id   = oi.order_id
    JOIN products p    ON oi.product_id = p.product_id
    LEFT JOIN category cat
           ON p.product_category_name = cat.product_category_name
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY category
    HAVING COUNT(*) >= 100
    ORDER BY one_star_pct DESC
    LIMIT 15
""")


# ----------------------------------------------------------------
# CELL 9 — QUERY 8: INSTALLMENT IMPACT ANALYSIS
# WHAT: Do installment customers spend more and review better?
# WHY:  Promote the right payment option to right customer
# HOW:  Push 6-10 installments for high-value categories
# ----------------------------------------------------------------
q8 = run_query("Q8: INSTALLMENT IMPACT ON AOV + REVIEWS", """
    SELECT
        p.payment_installments                               AS installments,
        COUNT(DISTINCT p.order_id)                           AS total_orders,
        ROUND(AVG(p.payment_value), 2)                       AS avg_order_value,
        ROUND(SUM(p.payment_value), 2)                       AS total_revenue,
        ROUND(AVG(r.review_score), 2)                        AS avg_review
    FROM payments p
    JOIN reviews r ON p.order_id = r.order_id
    WHERE p.payment_installments BETWEEN 1 AND 12
    GROUP BY p.payment_installments
    HAVING COUNT(DISTINCT p.order_id) >= 100
    ORDER BY p.payment_installments
""")


# ----------------------------------------------------------------
# CELL 10 — VISUALIZATIONS
# ----------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Chart 1 — Seller Grade Distribution
ax1 = axes[0, 0]
if 'seller_grade' in q2.columns:
    grade_counts = q2['seller_grade'].value_counts()
    colors_grade = {'TOP PERFORMER':'#2ecc71','AVERAGE':'#3498db','AT RISK':'#e74c3c'}
    bars = ax1.bar(grade_counts.index, grade_counts.values,
                   color=[colors_grade.get(g,'#95a5a6') for g in grade_counts.index])
    ax1.set_title('Seller Quality Distribution', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Number of Sellers')
    for bar, val in zip(bars, grade_counts.values):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                 str(val), ha='center', fontsize=11, fontweight='bold')
    ax1.grid(axis='y', linestyle='--', alpha=0.4)

# Chart 2 — Retention Rate by Month
ax2 = axes[0, 1]
q3_filtered = q3[q3['new_customers'] >= 100].copy()
ax2.bar(q3_filtered['cohort_month'], q3_filtered['retention_rate_pct'],
        color='steelblue', alpha=0.8)
ax2.plot(q3_filtered['cohort_month'], q3_filtered['retention_rate_pct'],
         color='navy', linewidth=2, marker='o', markersize=4)
ax2.set_title('Monthly Customer Retention Rate %', fontsize=12, fontweight='bold')
ax2.set_xlabel('Cohort Month')
ax2.set_ylabel('Retention Rate %')
ax2.tick_params(axis='x', rotation=45, labelsize=7)
ax2.grid(axis='y', linestyle='--', alpha=0.4)

# Chart 3 — Revenue per Customer by State (top 15)
ax3 = axes[1, 0]
q4_top = q4.head(15).sort_values('revenue_per_customer')
colors_state = ['#e74c3c' if v > q4['revenue_per_customer'].median()
                else '#3498db' for v in q4_top['revenue_per_customer']]
bars3 = ax3.barh(q4_top['state'], q4_top['revenue_per_customer'], color=colors_state)
ax3.set_title('Revenue per Customer by State\n(Red = Above Median)',
              fontsize=11, fontweight='bold')
ax3.set_xlabel('Revenue per Customer (R$)')
for bar, val in zip(bars3, q4_top['revenue_per_customer']):
    ax3.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
             f'R${val:.0f}', va='center', fontsize=8)
ax3.grid(axis='x', linestyle='--', alpha=0.4)

# Chart 4 — 1-Star Review by Category
ax4 = axes[1, 1]
q7_top = q7.head(10).sort_values('one_star_pct')
colors_1star = ['#e74c3c' if x > 15 else '#e67e22' for x in q7_top['one_star_pct']]
bars4 = ax4.barh(q7_top['category'], q7_top['one_star_pct'], color=colors_1star)
ax4.set_title('1-Star Review Rate by Category\n(Top 10 Worst)',
              fontsize=11, fontweight='bold')
ax4.set_xlabel('1-Star Review %')
for bar, val in zip(bars4, q7_top['one_star_pct']):
    ax4.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=8)
ax4.grid(axis='x', linestyle='--', alpha=0.4)

plt.suptitle('SQL Deep Analysis Dashboard', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/09_sql_dashboard.png', dpi=150)
plt.show()
print("Chart saved — outputs/09_sql_dashboard.png")

In [1]:
# ----------------------------------------------------------------
# CELL 11 — SAVE SQL QUERIES TO .sql FILE
# ----------------------------------------------------------------
sql_content = """
-- ================================================================
-- SQL ANALYSIS — OLIST E-COMMERCE
-- Project: E-Commerce Intelligence Platform
-- ================================================================

-- Q1: COMPLETE ORDER FUNNEL
SELECT
    order_status,
    COUNT(*)                                              AS order_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2)   AS pct_of_total,
    ROUND(SUM(p.payment_value), 2)                       AS revenue
FROM orders o
LEFT JOIN payments p ON o.order_id = p.order_id
GROUP BY order_status
ORDER BY order_count DESC;

-- Q2: SELLER QUALITY SCORECARD
SELECT
    s.seller_id,
    s.seller_state,
    COUNT(DISTINCT oi.order_id)                          AS total_orders,
    ROUND(SUM(oi.price), 2)                              AS total_revenue,
    ROUND(AVG(r.review_score), 2)                        AS avg_review,
    ROUND(AVG(julianday(o.order_delivered_customer_date)
          - julianday(o.order_purchase_timestamp)), 1)   AS avg_delivery_days,
    ROUND(SUM(CASE WHEN o.order_delivered_customer_date
                        > o.order_estimated_delivery_date
                   THEN 1.0 ELSE 0 END)
          * 100.0 / COUNT(*), 2)                         AS late_pct
FROM sellers s
JOIN order_items oi  ON s.seller_id  = oi.seller_id
JOIN orders o        ON oi.order_id  = o.order_id
JOIN reviews r       ON o.order_id   = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY s.seller_id, s.seller_state
HAVING COUNT(DISTINCT oi.order_id) >= 50
ORDER BY avg_review DESC;

-- Q3: CUSTOMER RETENTION RATE
WITH first_orders AS (
    SELECT
        customer_id,
        MIN(strftime('%Y-%m', order_purchase_timestamp)) AS first_month
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY customer_id
),
monthly_orders AS (
    SELECT DISTINCT
        o.customer_id,
        strftime('%Y-%m', o.order_purchase_timestamp) AS order_month
    FROM orders o
    WHERE o.order_status = 'delivered'
)
SELECT
    f.first_month,
    COUNT(DISTINCT f.customer_id)                        AS new_customers,
    COUNT(DISTINCT m.customer_id)                        AS returning_customers,
    ROUND(COUNT(DISTINCT m.customer_id) * 100.0
          / COUNT(DISTINCT f.customer_id), 2)            AS retention_rate_pct
FROM first_orders f
LEFT JOIN monthly_orders m
       ON f.customer_id = m.customer_id
      AND m.order_month > f.first_month
GROUP BY f.first_month
ORDER BY f.first_month;

-- Q4: REVENUE QUALITY BY STATE
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id)                           AS total_orders,
    ROUND(SUM(p.payment_value), 2)                       AS total_revenue,
    ROUND(SUM(p.payment_value)
          / COUNT(DISTINCT o.customer_id), 2)            AS revenue_per_customer,
    ROUND(AVG(r.review_score), 2)                        AS avg_review
FROM orders o
JOIN customers c  ON o.customer_id = c.customer_id
JOIN payments p   ON o.order_id    = p.order_id
JOIN reviews r    ON o.order_id    = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
HAVING COUNT(DISTINCT o.order_id) >= 100
ORDER BY revenue_per_customer DESC;

-- Q5: DELIVERY ROOT CAUSE
SELECT
    s.seller_state,
    c.customer_state,
    COUNT(*)                                             AS total_orders,
    ROUND(AVG(oi.product_weight_g), 0)                  AS avg_weight_g,
    ROUND(AVG(julianday(o.order_delivered_customer_date)
          - julianday(o.order_purchase_timestamp)), 1)   AS avg_days,
    ROUND(SUM(CASE WHEN o.order_delivered_customer_date
                        > o.order_estimated_delivery_date
                   THEN 1.0 ELSE 0 END)
          * 100.0 / COUNT(*), 2)                         AS late_pct
FROM orders o
JOIN order_items oi ON o.order_id    = oi.order_id
JOIN products p     ON oi.product_id = p.product_id
JOIN sellers s      ON oi.seller_id  = s.seller_id
JOIN customers c    ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY s.seller_state, c.customer_state
HAVING COUNT(*) >= 100
ORDER BY late_pct DESC;

-- Q6: 1-STAR REVIEW ROOT CAUSE
SELECT
    COALESCE(cat.product_category_name_english, 'Unknown') AS category,
    COUNT(*)                                               AS total_reviews,
    SUM(CASE WHEN r.review_score = 1 THEN 1 ELSE 0 END)   AS one_star_count,
    ROUND(SUM(CASE WHEN r.review_score = 1
                   THEN 1.0 ELSE 0 END)
          * 100.0 / COUNT(*), 2)                           AS one_star_pct,
    ROUND(AVG(r.review_score), 2)                          AS avg_review
FROM reviews r
JOIN orders o       ON r.order_id    = o.order_id
JOIN order_items oi ON o.order_id    = oi.order_id
JOIN products p     ON oi.product_id = p.product_id
LEFT JOIN category cat
       ON p.product_category_name = cat.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY category
HAVING COUNT(*) >= 100
ORDER BY one_star_pct DESC;
"""

with open('../sql/queries.sql', 'w') as f:
    f.write(sql_content)

print("SQL queries saved — sql/queries.sql")
print("\nNOTEBOOK 3 COMPLETE!")
print("=" * 50)
print("Queries executed  : 8")
print("Charts saved      : 1")
print("SQL file saved    : sql/queries.sql")

SQL queries saved — sql/queries.sql

NOTEBOOK 3 COMPLETE!
Queries executed  : 8
Charts saved      : 1
SQL file saved    : sql/queries.sql
